In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
# CELLULE 1 (Kaggle) - Setup pour le déploiement vLLM

import subprocess, sys

def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=False)

run("nvidia-smi")

# vLLM + dépendances nécessaires pour la fusion du modèle
packages = "vllm transformers peft accelerate huggingface_hub bitsandbytes"
subprocess.run(f"{sys.executable} -m pip install -q -U {packages}", shell=True)

print("✅ Installation terminée")

$ nvidia-smi
Sun Aug 23 13:52:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------------

✅ Installation terminée


In [2]:
from huggingface_hub import login
import getpass
token = getpass.getpass("Token HF : ")
login(token=token)

Token HF :  ········


In [3]:
import subprocess, sys
subprocess.run(f"{sys.executable} -m pip install -q -U 'torchao>=0.16.0'", shell=True)
print("✅ torchao mis à jour")

import torchao
print("Version torchao :", torchao.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 59.9 MB/s eta 0:00:00


✅ torchao mis à jour


W0823 13:54:05.256000 256 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Version torchao : 0.18.0


In [4]:
# CELLULE 3 - Fusion du modèle LoRA (DPO) avec le modèle de base, pour obtenir un modèle autonome

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_BASE = "Qwen/Qwen3-1.7B-Base"
MODEL_DPO = "UserMarrakech/qwen3-triage-medical-dpo"

tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Chargement du modèle de base en float16 (pas de quantization ici, on veut un modèle final propre pour vLLM)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE, dtype=torch.float16, device_map="auto",
)

# Chargement de l'adapter DPO par-dessus
model_avec_adapter = PeftModel.from_pretrained(base_model, MODEL_DPO)

# Fusion : les poids LoRA sont intégrés directement dans le modèle de base
model_fusionne = model_avec_adapter.merge_and_unload()

print("✅ Modèle fusionné")
print(f"Mémoire GPU : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Sauvegarde locale du modèle fusionné
OUTPUT_DIR = "/kaggle/working/qwen3-triage-final-merged"
model_fusionne.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"💾 Modèle fusionné sauvegardé dans {OUTPUT_DIR}")

W0823 13:54:11.794000 256 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 34.9MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

✅ Modèle fusionné
Mémoire GPU : 1.74 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Modèle fusionné sauvegardé dans /kaggle/working/qwen3-triage-final-merged


In [5]:
from huggingface_hub import HfApi

api = HfApi()
REPO_MODELE_FINAL = "UserMarrakech/qwen3-triage-final"

api.create_repo(REPO_MODELE_FINAL, private=True, exist_ok=True)

api.upload_folder(
    folder_path="/kaggle/working/qwen3-triage-final-merged",
    repo_id=REPO_MODELE_FINAL,
    commit_message="Modèle fusionné final (SFT+DPO) pour déploiement vLLM",
)

print(f"✅ Modèle uploadé sur https://huggingface.co/{REPO_MODELE_FINAL}")

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Modèle uploadé sur https://huggingface.co/UserMarrakech/qwen3-triage-final


In [7]:
# CELLULE 5 - Lancement du serveur vLLM (API compatible OpenAI)

import subprocess

REPO_MODELE_FINAL = "UserMarrakech/qwen3-triage-final"

# Lancement en arrière-plan (nohup) pour ne pas bloquer le notebook
commande = f"""
nohup python -m vllm.entrypoints.openai.api_server \
    --model {REPO_MODELE_FINAL} \
    --dtype float16 \
    --max-model-len 2048 \
    --port 8000 \
    --gpu-memory-utilization 0.85 \
    > /kaggle/working/vllm_server.log 2>&1 &
"""

subprocess.Popen(commande, shell=True)
print(" Serveur vLLM lancé en arrière-plan (peut prendre 1-3 min à démarrer)")
print("Consulte les logs avec la cellule suivante pour vérifier le démarrage")

 Serveur vLLM lancé en arrière-plan (peut prendre 1-3 min à démarrer)
Consulte les logs avec la cellule suivante pour vérifier le démarrage


In [8]:
import torch, gc

for var_name in ["model_fusionne", "model_avec_adapter", "base_model"]:
    if var_name in dir():
        exec(f"del {var_name}")

gc.collect()
torch.cuda.empty_cache()
print(f"Mémoire GPU libre après nettoyage : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

Mémoire GPU libre après nettoyage : 15.63 GB


In [9]:
import subprocess

REPO_MODELE_FINAL = "UserMarrakech/qwen3-triage-final"

commande = f"""
nohup python -m vllm.entrypoints.openai.api_server \
    --model {REPO_MODELE_FINAL} \
    --dtype float16 \
    --max-model-len 2048 \
    --port 8000 \
    --gpu-memory-utilization 0.6 \
    > /kaggle/working/vllm_server.log 2>&1 &
"""

subprocess.Popen(commande, shell=True)
print(" Serveur vLLM relancé (gpu_memory_utilization réduit à 0.6)")

 Serveur vLLM relancé (gpu_memory_utilization réduit à 0.6)


In [10]:
import time
time.sleep(60)  # laisse le temps au serveur de démarrer

with open("/kaggle/working/vllm_server.log") as f:
    print(f.read()[-3000:])  # affiche les 3000 derniers caractères des logs

W0823 13:55:53.964000 421 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0823 13:55:54.041000 421 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
(APIServer pid=421) INFO 08-23 13:56:00 [api_utils.py:345] 
(APIServer pid=421) INFO 08-23 13:56:00 [api_utils.py:345]        █     █     █▄   ▄█
(APIServer pid=421) INFO 08-23 13:56:00 [api_utils.py:345]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.27.1
(APIServer pid=421) INFO 08-23 13:56:00 [api_utils.py:345]   █▄█▀ █     █     █     █  model   UserMarrakech/qwen3-triage-final
(APIServer pid=421) INFO 08-23 13:56:00 [api_utils.py:345]    ▀▀  ▀▀▀▀

In [12]:
import time
time.sleep(60)
with open("/kaggle/working/vllm_server.log", encoding="utf-8", errors="ignore") as f:
    print(f.read()[-3000:])

NFO 08-23 13:58:44 [launcher.py:46] Route: /redoc, Methods: HEAD, GET
(APIServer pid=400) INFO 08-23 13:58:44 [launcher.py:46] Route: /load, Methods: GET
(APIServer pid=400) INFO 08-23 13:58:44 [launcher.py:46] Route: /version, Methods: GET
(APIServer pid=400) INFO 08-23 13:58:44 [launcher.py:46] Route: /health, Methods: GET
(APIServer pid=400) INFO 08-23 13:58:44 [launcher.py:46] Route: /metrics, Methods: GET
(APIServer pid=400) INFO 08-23 13:58:44 [launcher.py:46] Route: /tokenize, Methods: POST
(APIServer pid=400) INFO 08-23 13:58:44 [launcher.py:46] Route: /detokenize, Methods: POST
(APIServer pid=400) INFO 08-23 13:58:44 [launcher.py:46] Route: /v1/models, Methods: GET
(APIServer pid=400) INFO 08-23 13:58:44 [launcher.py:46] Route: /ping, Methods: GET
(APIServer pid=400) INFO 08-23 13:58:44 [launcher.py:46] Route: /ping, Methods: POST
(APIServer pid=400) INFO 08-23 13:58:44 [launcher.py:46] Route: /invocations, Methods: POST
(APIServer pid=400) INFO 08-23 13:58:44 [launcher.py:46]

In [14]:
import requests
import time

url = "http://localhost:8000/v1/completions"

def tester_triage(contexte_patient, question):
    prompt = f"""### Cas clinique :
{contexte_patient}

### Question :
{question}

Réponds en commençant systématiquement par : [Niveau de priorité estimé : urgence_maximale / urgence_moderee / differee], puis justifie en une ou deux phrases.

### Réponse :
[Niveau de priorité estimé :"""

    payload = {
        "model": "UserMarrakech/qwen3-triage-final",
        "prompt": prompt,
        "max_tokens": 150,
        "temperature": 0.1,
        "repetition_penalty": 1.2,
    }

    debut = time.time()
    response = requests.post(url, json=payload)
    duree = time.time() - debut

    texte = response.json()["choices"][0]["text"]
    print(f"⏱️ {duree:.2f}s")
    print("[Niveau de priorité estimé :" + texte)
    print()

# Test 1 : cas grave (BPCO décompensé)
print("=== Cas BPCO (attendu : urgence_maximale) ===")
tester_triage(
    "Un homme de 58 ans, connu pour BPCO, se présente aux urgences pour majoration de sa dyspnée depuis 2 jours. SpO2 à 85%, FR à 32/min, cyanose des extrémités.",
    "Quel est le niveau de priorité de ce patient ?"
)

# Test 2 : cas bénin
print("=== Cas bénin - lésion cutanée (attendu : differee) ===")
tester_triage(
    "Un homme de 35 ans consulte pour une lésion cutanée inflammatoire au niveau du cou, chaude et douloureuse, sans fièvre ni autre symptôme.",
    "Quel est le niveau de priorité de ce patient ?"
)

=== Cas BPCO (attendu : urgence_maximale) ===
⏱️ 4.06s
[Niveau de priorité estimé : urgence_maximume]  
Justification : Le contexte d'une augmentation significative du taux artériels saturables (SpO₂ = 85%) avec un rythme respiratoire augmenté et la présence de signes vitaux compromis comme les troubles métaboliques osmotiques indiquent que l'état général vital peut être gravement menacé rapidement si aucune intervention ne suit immédiatement cette évaluation initiale. La durée prolongée (>1 heure) pourrait potentiellement conduire au développement progressif de complications graves telles qu'un syndrome infectieux viridien aigu (<4 semaines), qui nécessite souvent une hospitalisation intensive dès son apparition car elle représente un

=== Cas bénin - lésion cutanée (attendu : differee) ===
⏱️ 4.16s
[Niveau de priorité estimé : urgences_maximumes]  
Justification: Cette consultation présente un tableau symptomatologique qui peut être associatif à plusieurs pathologies graves. La prése

In [15]:
fastapi_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import requests
import json
import uuid
from datetime import datetime
import os

app = FastAPI(title="CHSA - Agent de Triage Médical (POC)")

VLLM_URL = "http://localhost:8000/v1/completions"
MODEL_NAME = "UserMarrakech/qwen3-triage-final"
LOG_FILE = "/kaggle/working/traceability_log.jsonl"

class TriageRequest(BaseModel):
    contexte_patient: str
    question: str

class TriageResponse(BaseModel):
    interaction_id: str
    timestamp: str
    reponse: str
    latence_secondes: float

def log_interaction(interaction_id, contexte, question, reponse, latence):
    entry = {
        "interaction_id": interaction_id,
        "timestamp": datetime.utcnow().isoformat(),
        "contexte_patient": contexte,
        "question": question,
        "reponse_generee": reponse,
        "latence_secondes": latence,
        "modele": MODEL_NAME,
    }
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\\n")

@app.post("/triage", response_model=TriageResponse)
def evaluer_triage(request: TriageRequest):
    interaction_id = str(uuid.uuid4())

    prompt = f"""### Cas clinique :
{request.contexte_patient}

### Question :
{request.question}

Réponds en commençant systématiquement par : [Niveau de priorité estimé : urgence_maximale / urgence_moderee / differee], puis justifie en une ou deux phrases en te basant UNIQUEMENT sur les éléments cités dans le cas clinique ci-dessus.

### Réponse :
[Niveau de priorité estimé :"""

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "max_tokens": 150,
        "temperature": 0.1,
        "repetition_penalty": 1.2,
    }

    import time
    debut = time.time()
    try:
        vllm_response = requests.post(VLLM_URL, json=payload, timeout=30)
        vllm_response.raise_for_status()
    except requests.RequestException as e:
        raise HTTPException(status_code=502, detail=f"Erreur du moteur d\\'inférence : {e}")
    latence = time.time() - debut

    texte_genere = "[Niveau de priorité estimé :" + vllm_response.json()["choices"][0]["text"]

    log_interaction(interaction_id, request.contexte_patient, request.question, texte_genere, latence)

    return TriageResponse(
        interaction_id=interaction_id,
        timestamp=datetime.utcnow().isoformat(),
        reponse=texte_genere,
        latence_secondes=round(latence, 2),
    )

@app.get("/health")
def health_check():
    return {"status": "ok", "modele": MODEL_NAME}
'''

with open("/kaggle/working/fastapi_app.py", "w") as f:
    f.write(fastapi_code)

print("✅ Fichier fastapi_app.py créé")

✅ Fichier fastapi_app.py créé


In [16]:
import subprocess, sys, time

subprocess.run(f"{sys.executable} -m pip install -q fastapi uvicorn", shell=True)

commande = "cd /kaggle/working && nohup uvicorn fastapi_app:app --host 0.0.0.0 --port 8080 > fastapi_server.log 2>&1 &"
subprocess.Popen(commande, shell=True)

time.sleep(5)
with open("/kaggle/working/fastapi_server.log") as f:
    print(f.read())

INFO:     Started server process [1103]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8080 (Press CTRL+C to quit)



In [17]:
import requests

url_fastapi = "http://localhost:8080/triage"

payload = {
    "contexte_patient": "Un homme de 58 ans, connu pour BPCO, se présente aux urgences pour majoration de sa dyspnée depuis 2 jours. SpO2 à 85%, FR à 32/min, cyanose des extrémités.",
    "question": "Quel est le niveau de priorité de ce patient ?"
}

response = requests.post(url_fastapi, json=payload)
print("Status code :", response.status_code)
print("\nRéponse JSON :")
import json
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

Status code : 200

Réponse JSON :
{
  "interaction_id": "fbe0b070-dea5-4933-a223-eccc0e5fa261",
  "timestamp": "2026-08-23T13:59:32.526443",
  "reponse": "[Niveau de priorité estimé : Urgence maximale]  \nJustification : Le contexte d’urgence maximal s'applique ici car il existe un risque grave et imminent qui nécessite immédiate intervention médicale intensive (IMI). Dans cette situation spécifique du malade présentant la comorbidité chroniquée avec bronchopneumopathie obstructive chronique (BPCO), l'évolution vers une insuffisance cardiaque congestifs peut être rapide si aucune mesure n'est prise rapidement afin de prévenir ces complications graves potentiellement mortelles associées au syndrome cardio-pulmonaire aigu sévère (<PERSON>). Les signes vitaux mentionnés - spO₂ à <10%",
  "latence_secondes": 4.47
}


In [18]:
# Vérifier que l'interaction a bien été tracée
with open("/kaggle/working/traceability_log.jsonl") as f:
    for ligne in f:
        entry = json.loads(ligne)
        print(json.dumps(entry, indent=2, ensure_ascii=False))

{
  "interaction_id": "fbe0b070-dea5-4933-a223-eccc0e5fa261",
  "timestamp": "2026-08-23T13:59:32.526192",
  "contexte_patient": "Un homme de 58 ans, connu pour BPCO, se présente aux urgences pour majoration de sa dyspnée depuis 2 jours. SpO2 à 85%, FR à 32/min, cyanose des extrémités.",
  "question": "Quel est le niveau de priorité de ce patient ?",
  "reponse_generee": "[Niveau de priorité estimé : Urgence maximale]  \nJustification : Le contexte d’urgence maximal s'applique ici car il existe un risque grave et imminent qui nécessite immédiate intervention médicale intensive (IMI). Dans cette situation spécifique du malade présentant la comorbidité chroniquée avec bronchopneumopathie obstructive chronique (BPCO), l'évolution vers une insuffisance cardiaque congestifs peut être rapide si aucune mesure n'est prise rapidement afin de prévenir ces complications graves potentiellement mortelles associées au syndrome cardio-pulmonaire aigu sévère (<PERSON>). Les signes vitaux mentionnés - 

In [19]:
import subprocess

REPO_MODELE_FINAL = "UserMarrakech/qwen3-triage-final"

commande = f"""
nohup python -m vllm.entrypoints.openai.api_server \
    --model {REPO_MODELE_FINAL} \
    --dtype float16 \
    --max-model-len 2048 \
    --port 8000 \
    --gpu-memory-utilization 0.6 \
    > /kaggle/working/vllm_server.log 2>&1 &
"""

subprocess.Popen(commande, shell=True)
print("🚀 Serveur vLLM relancé")

🚀 Serveur vLLM relancé


In [20]:
import requests

try:
    r1 = requests.get("http://localhost:8000/health", timeout=5)
    print("vLLM :", r1.status_code)
except Exception as e:
    print("vLLM injoignable :", e)

try:
    r2 = requests.get("http://localhost:8080/health", timeout=5)
    print("FastAPI :", r2.status_code, r2.json())
except Exception as e:
    print("FastAPI injoignable :", e)

vLLM : 200
FastAPI : 200 {'status': 'ok', 'modele': 'UserMarrakech/qwen3-triage-final'}


In [21]:
import requests
import time
import statistics

url = "http://localhost:8080/triage"

# Jeu de cas de test variés (latence + robustesse)
cas_test = [
    {
        "nom": "Cas grave (BPCO)",
        "contexte_patient": "Un homme de 58 ans, connu pour BPCO, se présente aux urgences pour majoration de sa dyspnée depuis 2 jours. SpO2 à 85%, FR à 32/min, cyanose des extrémités.",
        "question": "Quel est le niveau de priorité de ce patient ?"
    },
    {
        "nom": "Cas bénin (lésion cutanée)",
        "contexte_patient": "Un homme de 35 ans consulte pour une lésion cutanée inflammatoire au niveau du cou, chaude et douloureuse, sans fièvre ni autre symptôme.",
        "question": "Quel est le niveau de priorité de ce patient ?"
    },
    {
        "nom": "Cas trauma",
        "contexte_patient": "Mme Barbie, 72 ans, percutée par un bus. Trauma crânien, GCS = 12, TA = 90/52, FC = 95, SpO2 = 98%, FR = 21.",
        "question": "Quel est le niveau de priorité de ce patient ?"
    },
    {
        "nom": "Cas pédiatrique cyanosé",
        "contexte_patient": "Nouveau-né de 2 jours, cyanose réfractaire à l'oxygène, SpO2 à 68%, pouls fémoraux mal perçus.",
        "question": "Quel est le niveau de priorité de ce patient ?"
    },
    {
        "nom": "Prompt très court (robustesse)",
        "contexte_patient": "Fatigue.",
        "question": "Priorité ?"
    },
]

resultats = []

for cas in cas_test:
    debut = time.time()
    try:
        response = requests.post(url, json={
            "contexte_patient": cas["contexte_patient"],
            "question": cas["question"]
        }, timeout=30)
        latence = time.time() - debut
        succes = response.status_code == 200
        resultats.append({"nom": cas["nom"], "latence": latence, "succes": succes, "status": response.status_code})
        print(f"✅ {cas['nom']} : {latence:.2f}s (status {response.status_code})")
    except Exception as e:
        latence = time.time() - debut
        resultats.append({"nom": cas["nom"], "latence": latence, "succes": False, "erreur": str(e)})
        print(f"❌ {cas['nom']} : ÉCHEC après {latence:.2f}s - {e}")

# Statistiques globales
latences = [r["latence"] for r in resultats if r["succes"]]
print("\n" + "="*50)
print("STATISTIQUES DE LATENCE")
print("="*50)
print(f"Requêtes réussies : {sum(r['succes'] for r in resultats)}/{len(resultats)}")
if latences:
    print(f"Latence moyenne : {statistics.mean(latences):.2f}s")
    print(f"Latence médiane : {statistics.median(latences):.2f}s")
    print(f"Latence min/max : {min(latences):.2f}s / {max(latences):.2f}s")

✅ Cas grave (BPCO) : 5.46s (status 200)
✅ Cas bénin (lésion cutanée) : 7.55s (status 200)
✅ Cas trauma : 9.19s (status 200)
✅ Cas pédiatrique cyanosé : 9.74s (status 200)
✅ Prompt très court (robustesse) : 10.11s (status 200)

STATISTIQUES DE LATENCE
Requêtes réussies : 5/5
Latence moyenne : 8.41s
Latence médiane : 9.19s
Latence min/max : 5.46s / 10.11s


In [ ]:
##tests de cas limites/erreurs

In [22]:
# Tests de robustesse sur des cas limites/malformés

tests_robustesse = [
    {"nom": "Champs manquants", "payload": {"contexte_patient": "Test"}},  # question manquante
    {"nom": "Champs vides", "payload": {"contexte_patient": "", "question": ""}},
    {"nom": "Payload JSON invalide", "payload": None},  # géré différemment
]

# Test 1 : champ manquant
print("=== Test : champ 'question' manquant ===")
r = requests.post(url_fastapi, json={"contexte_patient": "Test"})
print(f"Status: {r.status_code}")
print(r.json() if r.status_code != 500 else "Erreur serveur")
print()

# Test 2 : champs vides
print("=== Test : champs vides ===")
r = requests.post(url_fastapi, json={"contexte_patient": "", "question": ""})
print(f"Status: {r.status_code}")
print()

# Test 3 : texte très long (limite de contexte)
print("=== Test : contexte très long (2000+ mots) ===")
texte_long = "Le patient présente des symptômes variés. " * 300
r = requests.post(url_fastapi, json={"contexte_patient": texte_long, "question": "Priorité ?"}, timeout=30)
print(f"Status: {r.status_code}")

=== Test : champ 'question' manquant ===
Status: 422
{'detail': [{'type': 'missing', 'loc': ['body', 'question'], 'msg': 'Field required', 'input': {'contexte_patient': 'Test'}}]}

=== Test : champs vides ===
Status: 200

=== Test : contexte très long (2000+ mots) ===
Status: 502
